<a href="https://colab.research.google.com/github/mlbyhoang/marketing-attribution/blob/main/notebooks/08_gru_training_model1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 08 – Single-Layer GRU – Model Training

This notebook trains and evaluates the single-layer GRU using the selected hyperparameters.

In [ ]:
# Load Libraries
import ast
import keras_tuner as kt
import os
import json
import random
import warnings
from collections import Counter
from statistics import mode
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from imblearn.over_sampling import ADASYN, SMOTE
from imblearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from scipy.stats import loguniform
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import (
    auc,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    brier_score_loss,
    precision_recall_curve,
    precision_score,
    recall_score, roc_auc_score, roc_curve,
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import (
    GridSearchCV, RandomizedSearchCV, StratifiedKFold, train_test_split
)
from sklearn.preprocessing import (
    LabelEncoder, MultiLabelBinarizer, OneHotEncoder, StandardScaler
)
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.layers import (
    Input,
    Embedding,
    GRU,
    Dense,
    Lambda,
    Concatenate,
    Multiply,
    Layer,
    Softmax
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import Callback
from tensorflow.keras.metrics import AUC

warnings.filterwarnings('ignore')

In [ ]:
# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# Connect to Google Drive
drive.mount('/content/drive')

In [ ]:
# Paths
filtered_path = userdata.get("MTA_FILTERED_PATH")
save_path = userdata.get("MTA_RESULTS_PATH")

os.makedirs(save_path, exist_ok=True)

In [ ]:
# Load filtered journey data
df_filtered = pd.read_parquet(filtered_path)

print(f"Number of journeys: {len(df_filtered):,}")
print(f"Conversion rate: {df_filtered['order'].mean():.2%}")
print(f"Maximum journey length: {df_filtered['touchpoint_sum'].max()}")

In [ ]:
# Display first few rows
df_filtered.head()

In [ ]:
# Encode touchpoint sequences
all_touchpoints = [
    tp
    for seq in df_filtered["sequence"]
    for tp in seq
]

le = LabelEncoder()
le.fit(all_touchpoints)

print(f"Number of unique touchpoints: {len(le.classes_)}")
print(f"Touchpoint types: {le.classes_}")

# Reserve 0 for padding
X_seq_encoded = [
    le.transform(seq) + 1
    for seq in df_filtered["sequence"]
]

max_len = int(df_filtered["touchpoint_sum"].max())

X_seq_padded = np.array([
    np.pad(
        seq,
        (0, max_len - len(seq)),
        constant_values=0
    )
    for seq in X_seq_encoded
])

print(f"\nMaximum sequence length: {max_len}")
print(f"X_seq shape: {X_seq_padded.shape}")

# Sanity check
print(f"\nEncoded sequence:  {X_seq_padded[0]}")
print(f"Original sequence: {df_filtered['sequence'].iloc[0]}")

In [ ]:
# Define time features
time_feature_cols = [
    "interaction_hour_sin",
    "interaction_hour_cos",
    "interaction_weekday_sin",
    "interaction_weekday_cos",
    "tp_time_delta_min",
    "interaction_number",
    "is_click"
]

n_time_features = len(time_feature_cols)

# Build time feature sequences
def build_time_sequences(df, max_len, time_feature_cols):
    time_seqs = []

    for _, row in df.iterrows():

        time_vals = np.array([
            [d[col] for col in time_feature_cols]
            for d in row["seq_times"]
        ], dtype=np.float32)

        pad_len = max_len - len(time_vals)

        if pad_len > 0:
            time_vals = np.vstack([
                time_vals,
                np.zeros(
                    (pad_len, len(time_feature_cols)),
                    dtype=np.float32
                )
            ])
        else:
            time_vals = time_vals[:max_len]

        time_seqs.append(time_vals)

    return np.array(time_seqs, dtype=np.float32)


X_time = build_time_sequences(
    df_filtered,
    max_len,
    time_feature_cols
)

print(f"X_time shape: {X_time.shape}")

# Sanity check
print("\nFirst journey - first touchpoint:")
print(X_time[0, 0, :])

print("\nFirst journey - second time step:")
print(X_time[0, 1, :])

In [ ]:
# Numeric journey-level features and target
X_num = (
    df_filtered[["touchpoint_sum"]]
    .values
    .astype(np.float32)
)

y = (
    df_filtered["order"]
    .values
    .reshape(-1, 1)
    .astype(np.float32)
)

vocab_size = len(le.classes_) + 1
n_num_features = X_num.shape[1]

print(f"Vocabulary size:            {vocab_size}")
print(f"Number of time features:    {n_time_features}")
print(f"Number of numeric features: {n_num_features}")
print(f"Maximum sequence length:    {max_len}")

In [ ]:
# Train / validation / test split
indices = np.arange(len(df_filtered))

idx_trainval, idx_test = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=y.ravel()
)

idx_train, idx_val = train_test_split(
    idx_trainval,
    test_size=0.125,
    random_state=42,
    stratify=y[idx_trainval].ravel()
)

# Sequence input
X_seq_train, X_seq_val, X_seq_test = (
    X_seq_padded[idx_train],
    X_seq_padded[idx_val],
    X_seq_padded[idx_test]
)

# Time-feature input
X_time_train, X_time_val, X_time_test = (
    X_time[idx_train],
    X_time[idx_val],
    X_time[idx_test]
)

# Numerical journey-level input
X_num_train, X_num_val, X_num_test = (
    X_num[idx_train],
    X_num[idx_val],
    X_num[idx_test]
)

# Target
y_train, y_val, y_test = (
    y[idx_train],
    y[idx_val],
    y[idx_test]
)

print(f"Train:      {len(y_train)}")
print(f"Validation: {len(y_val)}")
print(f"Test:       {len(y_test)}")

print(f"\nConversion rate - Train:      {y_train.mean():.2%}")
print(f"Conversion rate - Validation: {y_val.mean():.2%}")
print(f"Conversion rate - Test:       {y_test.mean():.2%}")

In [ ]:
# Scale numerical features using training data only
scaler_num = StandardScaler()

X_num_train = scaler_num.fit_transform(X_num_train)
X_num_val = scaler_num.transform(X_num_val)
X_num_test = scaler_num.transform(X_num_test)

In [ ]:
# Oversample training data while keeping all model inputs aligned
def oversample_sequences(
    X_seq,
    X_time,
    X_num,
    y,
    sampling_strategy
):
    indices = np.arange(len(y)).reshape(-1, 1)

    ros = RandomOverSampler(
        sampling_strategy=sampling_strategy,
        random_state=42
    )

    indices_resampled, y_resampled = ros.fit_resample(
        indices,
        y.ravel()
    )

    indices_resampled = indices_resampled.ravel()

    return (
        X_seq[indices_resampled],
        X_time[indices_resampled],
        X_num[indices_resampled],
        y_resampled.reshape(-1, 1).astype(np.float32)
    )


In [ ]:
# Load selected imbalance parameters
save_path = userdata.get("MTA_RESULTS_PATH")

with open(
    f"{save_path}best_imbalance_params.json",
    "r"
) as f:
    best_params = json.load(f)

selected_class_weight = best_params["class_weight_1"]
selected_sampling_strategy = best_params["sampling_strategy"]

best_cw_dict = (
    {0: 1.0, 1: selected_class_weight}
    if selected_class_weight is not None
    else None
)

print("===== Selected Imbalance Parameters =====")
print(f"Class weights: {best_cw_dict}")
print(f"Sampling strategy: {selected_sampling_strategy}")

In [ ]:
# Apply oversampling to the training data only
X_seq_tr, X_time_tr, X_num_tr, y_tr = oversample_sequences(
    X_seq_train,
    X_time_train,
    X_num_train,
    y_train,
    sampling_strategy=selected_sampling_strategy
)

print("===== Training Data After Oversampling =====")
print(f"Samples: {len(y_tr):,}")
print(f"Conversion rate: {y_tr.mean():.2%}")
print("\nReady for hyperparameter tuning.")

In [ ]:
# Custom layers for the Dual-Attention GRU model
class AttentionSoftmax(tf.keras.layers.Layer):
    def call(self, inputs):
        x = tf.squeeze(inputs, axis=-1)
        x = tf.nn.softmax(x, axis=-1)
        return tf.expand_dims(x, axis=-1)


class WeightedSum(tf.keras.layers.Layer):
    def call(self, inputs):
        return tf.reduce_sum(inputs, axis=1)


class DualAttentionFusion(tf.keras.layers.Layer):
    def call(self, inputs):
        lam, v_i, v_c = inputs
        return (1 - lam) * v_i + lam * v_c


class ZeroStateLayer(tf.keras.layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]

        return [
            tf.zeros((batch_size, self.units)),
            tf.zeros((batch_size, 1))
        ]


class SliceLayer(tf.keras.layers.Layer):
    def __init__(self, start, end, **kwargs):
        super().__init__(**kwargs)
        self.start = start
        self.end = end

    def call(self, inputs):
        return inputs[:, :, self.start:self.end]


class DecoderGRUCell(tf.keras.layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units

        self.gru_cell = tf.keras.layers.GRUCell(
            units,
            dropout=0.0,
            recurrent_dropout=0.0,
            name="decoder_gru_cell"
        )

        self.click_dense = tf.keras.layers.Dense(
            1,
            activation="sigmoid",
            name="click_dense"
        )

    @property
    def state_size(self):
        return [self.units, 1]

    @property
    def output_size(self):
        return self.units + 1

    def call(self, inputs, states, training=False):
        h_last = inputs
        s_prev = states[0]
        z_prev = states[1]

        gru_input = tf.concat(
            [z_prev, h_last],
            axis=-1
        )

        s_j, [s_j_out] = self.gru_cell(
            gru_input,
            [s_prev],
            training=training
        )

        z_hat_j = self.click_dense(
            tf.concat(
                [z_prev, s_j],
                axis=-1
            )
        )

        output = tf.concat(
            [s_j, z_hat_j],
            axis=-1
        )

        return output, [s_j, z_hat_j]


print("Custom layers and DecoderGRUCell defined.")

In [ ]:
# Hyperparameter tuning for Model 3: Dual-Attention GRU
def build_ablation_3_tuning(hp):
    embedding_dim = hp.Choice(
        "embedding_dim",
        [8, 16, 32]
    )

    gru_units = hp.Choice(
        "gru_units",
        [16, 32, 64]
    )

    dropout = hp.Float(
        "dropout",
        min_value=0.1,
        max_value=0.4,
        step=0.1
    )

    learning_rate = hp.Choice(
        "learning_rate",
        [1e-3, 5e-4, 1e-4]
    )

    dense_units = hp.Choice(
        "dense_units",
        [8, 16, 32]
    )

    click_weight = hp.Choice(
        "click_weight",
        [0.1, 0.3, 0.5]
    )

    tf.keras.backend.clear_session()

    # Model inputs
    seq_input = Input(
        shape=(max_len,),
        name="seq_input"
    )

    time_input = Input(
        shape=(max_len, n_time_features),
        name="time_input"
    )

    num_input = Input(
        shape=(n_num_features,),
        name="num_input"
    )

    # Touchpoint embedding
    embedding = Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        mask_zero=True,
        name="embedding"
    )(seq_input)

    # Repeat journey-level numerical features across all time steps
    num_repeated = tf.keras.layers.RepeatVector(
        max_len,
        name="num_repeated"
    )(num_input)

    # Combine sequential inputs
    x = Concatenate(
        axis=-1,
        name="combined_input"
    )([
        embedding,
        time_input,
        num_repeated
    ])

    # Encoder GRU
    encoder_out = GRU(
        gru_units,
        return_sequences=True,
        return_state=True,
        dropout=dropout,
        recurrent_dropout=dropout / 2,
        name="encoder"
    )(x)

    encoder_sequences = encoder_out[0]
    h_last = encoder_out[1]

    # Decoder input
    h_last_repeated = tf.keras.layers.RepeatVector(
        max_len,
        name="h_last_repeated"
    )(h_last)

    initial_states = ZeroStateLayer(
        gru_units,
        name="zero_state"
    )(seq_input)

    s_0 = initial_states[0]
    z_0 = initial_states[1]

    decoder_rnn = tf.keras.layers.RNN(
        DecoderGRUCell(
            units=gru_units,
            name="decoder_cell"
        ),
        return_sequences=True,
        name="decoder_rnn"
    )

    decoder_out = decoder_rnn(
        h_last_repeated,
        initial_state=[s_0, z_0]
    )

    decoder_sequences = SliceLayer(
        0,
        gru_units,
        name="decoder_sequences"
    )(decoder_out)

    click_output = SliceLayer(
        gru_units,
        gru_units + 1,
        name="decoder"
    )(decoder_out)

    # Impression-side attention
    energy_imp = Dense(
        gru_units // 2,
        activation="tanh",
        name="energy_imp"
    )(encoder_sequences)

    score_imp = Dense(
        1,
        name="score_imp"
    )(energy_imp)

    a_imp = AttentionSoftmax(
        name="a_imp"
    )(score_imp)

    weighted_imp = Multiply(
        name="weighted_imp"
    )([
        encoder_sequences,
        a_imp
    ])

    v_imp = WeightedSum(
        name="v_imp"
    )(weighted_imp)

    # Click-side attention
    energy_clk = Dense(
        gru_units // 2,
        activation="tanh",
        name="energy_clk"
    )(decoder_sequences)

    score_clk = Dense(
        1,
        name="score_clk"
    )(energy_clk)

    a_clk = AttentionSoftmax(
        name="a_clk"
    )(score_clk)

    weighted_clk = Multiply(
        name="weighted_clk"
    )([
        decoder_sequences,
        a_clk
    ])

    v_clk = WeightedSum(
        name="v_clk"
    )(weighted_clk)

    # Dual-attention fusion
    lambda_input = Concatenate(
        name="lambda_input"
    )([
        v_imp,
        v_clk
    ])

    lambda_val = Dense(
        1,
        activation="sigmoid",
        name="lambda_val"
    )(lambda_input)

    final_input = DualAttentionFusion(
        name="dual_attention_fusion"
    )([
        lambda_val,
        v_imp,
        v_clk
    ])

    # Conversion prediction
    final_hidden = Dense(
        dense_units,
        activation="relu",
        name="final_hidden"
    )(final_input)

    conv_output = Dense(
        1,
        activation="sigmoid",
        name="conversion"
    )(final_hidden)

    model = Model(
        inputs=[
            seq_input,
            time_input,
            num_input
        ],
        outputs=[
            click_output,
            conv_output
        ],
        name="dual_attention_gru"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss={
            "decoder": "binary_crossentropy",
            "conversion": "binary_crossentropy"
        },
        loss_weights={
            "decoder": click_weight,
            "conversion": 1.0
        },
        metrics={
            "decoder": [
                AUC(name="click_auc")
            ],
            "conversion": [
                AUC(name="roc_auc"),
                AUC(name="pr_auc", curve="PR")
            ]
        }
    )

    return model


print("Dual-Attention GRU tuning model defined.")

In [ ]:
# Configure Bayesian hyperparameter optimization
tuner_abl3 = kt.BayesianOptimization(
    build_ablation_3_tuning,
    objective=kt.Objective(
        "val_conversion_pr_auc",
        direction="max"
    ),
    max_trials=20,
    directory="tuner_results",
    project_name="ablation_3_tuning",
    overwrite=True
)

print("===== Tuning Model 3: Dual-Attention GRU =====")
print("Joint loss: click_weight × decoder loss + conversion loss")

tuner_abl3.search(
    [X_seq_tr, X_time_tr, X_num_tr],
    {
        "decoder": y_click_tr,
        "conversion": y_tr
    },
    validation_data=(
        [X_seq_val, X_time_val, X_num_val],
        {
            "decoder": y_click_val,
            "conversion": y_val
        }
    ),
    epochs=30,
    batch_size=64,
    callbacks=[
        EarlyStopping(
            monitor="val_conversion_pr_auc",
            patience=5,
            mode="max",
            restore_best_weights=True
        )
    ],
    verbose=1
)

In [ ]:
# Retrieve best hyperparameters
best_hps_abl3 = tuner_abl3.get_best_hyperparameters(
    num_trials=1
)[0]

hp_abl3 = {
    "embedding_dim": best_hps_abl3.get("embedding_dim"),
    "gru_units": best_hps_abl3.get("gru_units"),
    "dropout": best_hps_abl3.get("dropout"),
    "learning_rate": best_hps_abl3.get("learning_rate"),
    "dense_units": best_hps_abl3.get("dense_units"),
    "click_weight": best_hps_abl3.get("click_weight")
}


# Save best hyperparameters
with open(
    f"{save_path}best_hp_ablation_3.json",
    "w"
) as f:
    json.dump(
        hp_abl3,
        f,
        indent=4
    )


# Print best hyperparameters
print("\n===== Best Hyperparameters: Model 3 =====")

for key, value in hp_abl3.items():
    print(f"{key}: {value}")

print(
    "\nBest hyperparameters saved to "
    "best_hp_ablation_3.json"
)